In [1]:
!pip install ucimlrepo

In [3]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

pd.set_option('display.max_columns', None)
np.random.seed(42)

In [4]:
steel = fetch_ucirepo(id=198)

X = steel.data.features
y_raw = steel.data.targets

print("Features:", X.shape)
print("Targets :", y_raw.shape)

Features: (1941, 27)
Targets : (1941, 7)


In [5]:
# Are the seven fault flags mutually exclusive?
flags_per_row = y_raw.sum(axis=1)
print("Flags per row:\n", flags_per_row.value_counts(), "\n")

y = y_raw.idxmax(axis=1)
y.name = "Fault_Type"

print("Class distribution:")
print(y.value_counts(), "\n")
print("As proportions:")
print((y.value_counts(normalize=True) * 100).round(1))

Flags per row:
 1    1941
Name: count, dtype: int64 

Class distribution:
Fault_Type
Other_Faults    673
Bumps           402
K_Scratch       391
Z_Scratch       190
Pastry          158
Stains           72
Dirtiness        55
Name: count, dtype: int64 

As proportions:
Fault_Type
Other_Faults    34.7
Bumps           20.7
K_Scratch       20.1
Z_Scratch        9.8
Pastry           8.1
Stains           3.7
Dirtiness        2.8
Name: proportion, dtype: float64


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, " Test:", X_test.shape, "\n")

comparison = pd.DataFrame({
    "train_%": (y_train.value_counts(normalize=True) * 100).round(1),
    "test_%":  (y_test.value_counts(normalize=True) * 100).round(1),
    "test_n":   y_test.value_counts()
})
print(comparison)

Train: (1552, 27)  Test: (389, 27) 

              train_%  test_%  test_n
Fault_Type                           
Other_Faults     34.7    34.7     135
Bumps            20.7    20.8      81
K_Scratch        20.2    20.1      78
Z_Scratch         9.8     9.8      38
Pastry            8.1     8.2      32
Stains            3.7     3.6      14
Dirtiness         2.8     2.8      11


In [7]:
test_df = X_test.copy()
test_df["Fault_Type"] = y_test
test_df.to_csv("test_data.csv", index=False)

print("Saved test_data.csv:", test_df.shape)
print(test_df["Fault_Type"].value_counts())

Saved test_data.csv: (389, 28)
Fault_Type
Other_Faults    135
Bumps            81
K_Scratch        78
Z_Scratch        38
Pastry           32
Stains           14
Dirtiness        11
Name: count, dtype: int64


In [8]:
scales = pd.DataFrame({
    "min": X_train.min(),
    "max": X_train.max(),
    "range": X_train.max() - X_train.min(),
    "has_negative": (X_train.min() < 0)
}).sort_values("range", ascending=False)

print(scales)
print("\nFeatures with negative values:", scales["has_negative"].sum())

                             min           max         range  has_negative
Y_Maximum              6724.0000  1.298769e+07  1.298097e+07         False
Y_Minimum              6712.0000  1.298766e+07  1.298095e+07         False
Sum_of_Luminosity       250.0000  3.918209e+06  3.917959e+06         False
Pixels_Areas              2.0000  3.733400e+04  3.733200e+04         False
X_Maximum                 4.0000  1.713000e+03  1.709000e+03         False
X_Minimum                 0.0000  1.705000e+03  1.705000e+03         False
X_Perimeter               2.0000  1.275000e+03  1.273000e+03         False
Y_Perimeter               1.0000  9.030000e+02  9.020000e+02         False
Length_of_Conveyer     1227.0000  1.794000e+03  5.670000e+02         False
Steel_Plate_Thickness    40.0000  3.000000e+02  2.600000e+02         False
Maximum_of_Luminosity    37.0000  2.530000e+02  2.160000e+02         False
Minimum_of_Luminosity     0.0000  2.030000e+02  2.030000e+02         False
LogOfAreas               

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "Decision Tree": Pipeline([
        ("clf", DecisionTreeClassifier(random_state=42))
    ]),
    "kNN": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=5))
    ]),
    "Naive Bayes": Pipeline([
        ("clf", GaussianNB())
    ]),
    "Random Forest (Ensemble)": Pipeline([
        ("clf", RandomForestClassifier(n_estimators=200, random_state=42))
    ]),
}

for name, pipe in models.items():
    steps = " → ".join(s[0] for s in pipe.steps)
    print(f"{name:26s} : {steps}")

Logistic Regression        : scaler → clf
Decision Tree              : clf
kNN                        : scaler → clf
Naive Bayes                : clf
Random Forest (Ensemble)   : clf


In [10]:
import time
from sklearn.metrics import accuracy_score

fitted = {}

for name, pipe in models.items():
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    elapsed = time.perf_counter() - t0
    fitted[name] = pipe

    train_acc = accuracy_score(y_train, pipe.predict(X_train))
    test_acc  = accuracy_score(y_test,  pipe.predict(X_test))

    print(f"{name:26s} train={train_acc:.3f}  test={test_acc:.3f}  "
          f"gap={train_acc - test_acc:+.3f}  ({elapsed:.2f}s)")

Logistic Regression        train=0.734  test=0.728  gap=+0.006  (0.12s)
Decision Tree              train=1.000  test=0.743  gap=+0.257  (0.02s)
kNN                        train=0.816  test=0.728  gap=+0.088  (0.00s)
Naive Bayes                train=0.467  test=0.452  gap=+0.015  (0.00s)
Random Forest (Ensemble)   train=1.000  test=0.799  gap=+0.201  (0.56s)


In [11]:
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

def evaluate(pipe, X_te, y_te, average="macro"):
    y_pred  = pipe.predict(X_te)
    y_proba = pipe.predict_proba(X_te)
    return {
        "Accuracy":  accuracy_score(y_te, y_pred),
        "AUC":       roc_auc_score(y_te, y_proba, multi_class="ovr",
                                   average=average, labels=pipe.classes_),
        "Precision": precision_score(y_te, y_pred, average=average, zero_division=0),
        "Recall":    recall_score(y_te, y_pred, average=average, zero_division=0),
        "F1":        f1_score(y_te, y_pred, average=average, zero_division=0),
        "MCC":       matthews_corrcoef(y_te, y_pred),
    }

results_macro = pd.DataFrame(
    {name: evaluate(p, X_test, y_test) for name, p in fitted.items()}
).T.round(4)

print(results_macro)

                          Accuracy     AUC  Precision  Recall      F1     MCC
Logistic Regression         0.7275  0.9379     0.7613  0.7305  0.7418  0.6487
Decision Tree               0.7429  0.8534     0.7531  0.7559  0.7530  0.6688
kNN                         0.7275  0.9184     0.7458  0.7482  0.7416  0.6548
Naive Bayes                 0.4524  0.8386     0.4315  0.4795  0.3830  0.3789
Random Forest (Ensemble)    0.7995  0.9640     0.8452  0.7793  0.8081  0.7400


In [12]:
from sklearn.metrics import classification_report

best = fitted["Random Forest (Ensemble)"]
print(classification_report(y_test, best.predict(X_test), digits=3))

results_weighted = pd.DataFrame(
    {name: evaluate(p, X_test, y_test, average="weighted") for name, p in fitted.items()}
).T.round(4)
print("\nWeighted averaging:\n", results_weighted)

              precision    recall  f1-score   support

       Bumps      0.730     0.667     0.697        81
   Dirtiness      0.889     0.727     0.800        11
   K_Scratch      0.973     0.936     0.954        78
Other_Faults      0.711     0.837     0.769       135
      Pastry      0.720     0.562     0.632        32
      Stains      0.923     0.857     0.889        14
   Z_Scratch      0.971     0.868     0.917        38

    accuracy                          0.799       389
   macro avg      0.845     0.779     0.808       389
weighted avg      0.806     0.799     0.799       389


Weighted averaging:
                           Accuracy     AUC  Precision  Recall      F1     MCC
Logistic Regression         0.7275  0.9066     0.7329  0.7275  0.7277  0.6487
Decision Tree               0.7429  0.8283     0.7455  0.7429  0.7429  0.6688
kNN                         0.7275  0.8941     0.7318  0.7275  0.7245  0.6548
Naive Bayes                 0.4524  0.7955     0.5741  0.4524  0.4003

In [13]:
import joblib, json, os, sklearn

os.makedirs("model", exist_ok=True)

def slug(name):
    return (name.lower().replace(" (ensemble)", "").replace(" ", "_"))

for name, pipe in fitted.items():
    joblib.dump(pipe, f"model/{slug(name)}.joblib")
    print("saved:", f"model/{slug(name)}.joblib")

metadata = {
    "feature_names": list(X_train.columns),
    "class_names":   sorted(y.unique().tolist()),
    "models":        {slug(n): n for n in fitted},
    "sklearn_version": sklearn.__version__,
}
with open("model/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

results_macro.to_csv("model/results_macro.csv")
results_weighted.to_csv("model/results_weighted.csv")

print("\nsklearn :", sklearn.__version__)
print("joblib  :", joblib.__version__)
print("pandas  :", pd.__version__)
print("numpy   :", np.__version__)

saved: model/logistic_regression.joblib
saved: model/decision_tree.joblib
saved: model/knn.joblib
saved: model/naive_bayes.joblib
saved: model/random_forest.joblib

sklearn : 1.7.2
joblib  : 1.5.2
pandas  : 2.3.3
numpy   : 2.3.5


In [14]:
import os
print(os.getcwd())
print()
for item in sorted(os.listdir()):
    print(item)

/Users/asmitabera/steel-plate-fault-classifier

.ipynb_checkpoints
Untitled.ipynb
model
test_data.csv
